In [329]:
import pandas as pd
import numpy as np
from matplotlib.pyplot import plot

In [330]:
house_prices = pd.read_csv("data/Housing.csv")

### Since there are non-numeric from the dataset we must first convert em into numbers

In [331]:
house_prices.head(10)

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished
5,10850000,7500,3,3,1,yes,no,yes,no,yes,2,yes,semi-furnished
6,10150000,8580,4,3,4,yes,no,no,no,yes,2,yes,semi-furnished
7,10150000,16200,5,3,2,yes,no,no,no,no,0,no,unfurnished
8,9870000,8100,4,1,2,yes,yes,yes,no,yes,2,yes,furnished
9,9800000,5750,3,2,4,yes,yes,no,no,yes,1,yes,unfurnished


In [332]:
non_numeric_cols = ['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'prefarea']

house_prices[non_numeric_cols] = house_prices[non_numeric_cols].apply(lambda col: col.map({'yes': 1, 'no': 0}))

house_prices = pd.get_dummies(house_prices, columns=['furnishingstatus'],
                              drop_first=True)  # Drop the first furnishingstatus, it will become the baseline later
bool_cols = house_prices.select_dtypes(include='bool').columns
house_prices[bool_cols] = house_prices[bool_cols].astype(int)
house_prices

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus_semi-furnished,furnishingstatus_unfurnished
0,13300000,7420,4,2,3,1,0,0,0,1,2,1,0,0
1,12250000,8960,4,4,4,1,0,0,0,1,3,0,0,0
2,12250000,9960,3,2,2,1,0,1,0,0,2,1,1,0
3,12215000,7500,4,2,2,1,0,1,0,1,3,1,0,0
4,11410000,7420,4,1,2,1,1,1,0,1,2,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
540,1820000,3000,2,1,1,1,0,1,0,0,2,0,0,1
541,1767150,2400,3,1,1,0,0,0,0,0,0,0,1,0
542,1750000,3620,2,1,1,1,0,0,0,0,0,0,0,1
543,1750000,2910,3,1,1,0,0,0,0,0,0,0,0,0


### Now separate the training data from the label data, the price is the label

In [333]:
print(house_prices.isnull().sum())  # Check if there are any null values first, since wala then proceed

price                              0
area                               0
bedrooms                           0
bathrooms                          0
stories                            0
mainroad                           0
guestroom                          0
basement                           0
hotwaterheating                    0
airconditioning                    0
parking                            0
prefarea                           0
furnishingstatus_semi-furnished    0
furnishingstatus_unfurnished       0
dtype: int64


In [334]:
# X train
X = house_prices.iloc[:, 1:].values
# y true
y = house_prices.iloc[:, 0].values
# Split the data for training and testing ( same sa digit recognition system ko )
# 80% of the data will be used for the training and the rest is for testing
shuffle_idx = np.random.permutation(len(X))
X = X[shuffle_idx]
y = y[shuffle_idx]

train_percent_index = int(0.8 * len(X))
X_train, X_test = X[:train_percent_index], X[train_percent_index:]
y_train, y_test = y[:train_percent_index], y[train_percent_index:]

# Scale not Normalize
# Kasi tangina naga explode gradients since there are scenarios where the input are smaller than the label
# And once the grads are updated it just explodes ( the weight are large as fuck )
# That's why when you try to compute for the loss umaabot ng trillions
X_mean = X_train.mean(axis=0)
X_std = X_train.std(axis=0)

y_mean = y_train.mean()
y_std = y_train.std()

X_train = (X_train - X_mean) / X_std
X_test = (X_test - X_mean) / X_std

y_train = (y_train - y_mean) / y_std
y_test = (y_test - y_mean) / y_std

# Shuffle timee
shuffle_idx = np.random.permutation(len(X_train))
X_train = X_train[shuffle_idx]
y_train = y_train[shuffle_idx]
X_train


array([[-0.48570068,  0.040821  , -0.56786585, ..., -0.54908336,
         1.19822961, -0.70589078],
       [-1.13977821,  0.040821  , -0.56786585, ..., -0.54908336,
        -0.83456459,  1.41664976],
       [ 1.01149149, -1.32825269, -0.56786585, ..., -0.54908336,
         1.19822961, -0.70589078],
       ...,
       [-0.61340371, -1.32825269, -0.56786585, ..., -0.54908336,
        -0.83456459,  1.41664976],
       [-1.03450331,  0.040821  , -0.56786585, ..., -0.54908336,
        -0.83456459,  1.41664976],
       [ 0.3935736 ,  1.4098947 ,  3.49097856, ..., -0.54908336,
         1.19822961, -0.70589078]], shape=(436, 13))

In [335]:
learning_rate = 0.1
W = []
B = []

layer_sizes = [13, 16, 1]  # input, hidden, output
np.shape(X_train)
# Init Weights and Biases
for n in range(len(layer_sizes) - 1):
    _in = layer_sizes[n]
    _out = layer_sizes[n + 1]
    W.append(np.random.randn(_in, _out) * np.sqrt(2.0 / _in))
    B.append(np.zeros((1, _out)))

Z = [None] * (len(layer_sizes) - 1)  # Summation aft mat multiplication is applied
H = [None] * (len(layer_sizes) - 1)  # Activation
len(W)
# self.__W1 = np.random.randn(784, 256) * np.sqrt(2.0 / 784)
# self.__B1 = np.zeros((1, 256))

2

In [336]:
def relu(x_array):
    return np.maximum(0, x_array)


def unscale_y(y_scaled):
    """ The scaled was [-3, 3] lang so unscale para mabalik sa orig scale """
    return (y_scaled * y_std) + y_mean


def relu_derivative(z):
    return (z > 0).astype(float)


def MSE(y_pred, y_true):
    return np.square(np.subtract(y_true, y_pred)).mean()


def MSE_derivative(y_true, y_pred):  # Na-cow
    """
    Calculates the gradient of MSE with respect to y_pred.
    Formula: 2/n * (y_pred - y_true)
    """
    y_true = y_true.reshape(y_pred.shape)
    n = y_true.size
    return 2 * (y_pred - y_true) / n


def evaluate():
    y_pred_scaled = forward_pass(X_test)
    y_pred = unscale_y(y_pred_scaled.ravel())
    y_true = unscale_y(y_test)

    rmse = np.sqrt(np.mean((y_pred - y_true) ** 2))
    mae = np.mean(np.abs(y_pred - y_true))

    total_squared_error = np.sum((y_true - y_pred) ** 2) # How wrong our model is
    dumb_guessed_tse = np.sum((y_true - np.mean(y_true)) ** 2) # How wrong the "guessed mean"
    r2 = 1 - (total_squared_error / dumb_guessed_tse)
    return rmse, mae, r2


def update_params(grads):
    w, b = grads
    for i in (range(len(W))):
        W[i] -= w[i] * learning_rate
        B[i] -= b[i] * learning_rate


def forward_pass(X_batch):
    h = X_batch
    for i in range(len(W)):
        Z[i] = h @ W[i] + B[i]
        H[i] = relu(Z[i]) if i < len(W) - 1 else Z[i]
        h = H[i]
    return h  # y_pred


def back_propagation(X_batch, y_pred, y_true):
    """ There is probably a simpler/better way of doing this but since I only had 5 hours of sleep I won't overthink it. """
    dL_dy = MSE_derivative(y_true, y_pred)

    dL_dz = dL_dy  # Pass the y_pred; could've passed the y_pred var instead of the Z but it's just the same
    dL_dW = [None] * len(W)
    dL_dB = [None] * len(W)
    # Start the loop from the max since ts is not forward pass
    for i in reversed(range(len(W))):
        input_h = X_batch if i == 0 else H[i - 1]  # Input activation
        dL_dW[i] = (input_h.T @ dL_dz) / X_batch.shape[0]
        dL_dB[i] = np.sum(dL_dz, axis=0, keepdims=True) / X_batch.shape[0]
        if i > 0:
            dL_dz = (dL_dz @ W[i].T) * relu_derivative(Z[i - 1])
    return dL_dW, dL_dB


def train(epochs=200, batch_size=64):
    n_samples = X_train.shape[0]
    n_batches = n_samples // batch_size

    for e in range(epochs):
        indices = np.random.permutation(n_samples)
        X_shuffled = X_train[indices]
        y_shuffled = y_train[indices]

        losses = 0
        for batch in range(n_batches):
            start_idx = batch * batch_size
            end_idx = start_idx + batch_size

            X_batch = X_shuffled[start_idx:end_idx]
            y_batch = y_shuffled[start_idx:end_idx]

            y_hat = forward_pass(X_batch)

            losses += MSE(y_hat, y_batch,)

            grads = back_propagation(X_batch, y_hat, y_batch)
            update_params(grads)
        losses_mean = losses / n_batches
        rmse, mae, r2 = evaluate()

        print(
            f"Epoch {e + 1}/{epochs} - "
            f"Loss: {losses_mean:.4f} - "
            f"RMSE: {rmse:,.0f} - "
            f"MAE: {mae:,.0f} - "
            f"R2: {r2:.4f} - "
        )

In [337]:
train()

Epoch 1/200 - Loss: 2.1218 - RMSE: 1,528,562 - MAE: 1,167,493 - R2: 0.2695 - 
Epoch 2/200 - Loss: 2.0314 - RMSE: 1,497,964 - MAE: 1,141,592 - R2: 0.2985 - 
Epoch 3/200 - Loss: 1.9091 - RMSE: 1,474,040 - MAE: 1,120,290 - R2: 0.3207 - 
Epoch 4/200 - Loss: 1.9219 - RMSE: 1,449,548 - MAE: 1,099,824 - R2: 0.3431 - 
Epoch 5/200 - Loss: 1.9330 - RMSE: 1,429,639 - MAE: 1,082,421 - R2: 0.3610 - 
Epoch 6/200 - Loss: 1.9444 - RMSE: 1,411,901 - MAE: 1,067,415 - R2: 0.3768 - 
Epoch 7/200 - Loss: 1.8582 - RMSE: 1,394,618 - MAE: 1,052,189 - R2: 0.3919 - 
Epoch 8/200 - Loss: 1.9396 - RMSE: 1,380,329 - MAE: 1,040,944 - R2: 0.4043 - 
Epoch 9/200 - Loss: 1.8569 - RMSE: 1,367,290 - MAE: 1,031,026 - R2: 0.4155 - 
Epoch 10/200 - Loss: 1.8730 - RMSE: 1,354,838 - MAE: 1,021,961 - R2: 0.4261 - 
Epoch 11/200 - Loss: 1.8689 - RMSE: 1,343,914 - MAE: 1,013,730 - R2: 0.4353 - 
Epoch 12/200 - Loss: 1.8581 - RMSE: 1,333,961 - MAE: 1,005,121 - R2: 0.4437 - 
Epoch 13/200 - Loss: 1.7458 - RMSE: 1,325,379 - MAE: 997,790 